# Training BLIP-2 — Image Captioning Aksara Lontara
## Model: `Salesforce/blip2-opt-2.7b`

Notebook ini menjalankan **seluruh pipeline** dalam satu proses (mengikuti pola `training_blip.ipynb`):
1. Mount Google Drive & Install Library
2. Pipeline: Persiapan Dataset → Training → Evaluasi (metrik + akurasi per-karakter) → Inference & Visualisasi → Simpan ke Drive

**Konfigurasi:** Batch size 4 · Epochs 20 · LR 2e-5 · AdamW · 8-bit quantization · Gradient checkpointing · Freeze ViT & LLM · MAX_LENGTH 128 · split per-caption (sama dgn referensi BLIP)

> **Pastikan runtime menggunakan GPU** (Runtime → Change runtime type → GPU).
> Semua `import` berada di dalam sel Pipeline agar sel dapat dijalankan mandiri (tahan terhadap reset runtime).

---
## 1. Setup: Mount Google Drive & Install Library

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SETUP: Mount Google Drive & Install Library
# ═══════════════════════════════════════════════════════════════
import os
try:
    os.chdir("/content")
except OSError:
    pass

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

# Copy dataset dari Drive ke local (lebih stabil & cepat)
assert os.path.exists(DRIVE_DIR), f"ERROR: {DRIVE_DIR} tidak ditemukan!"
!rm -rf {LOCAL_DIR}
!cp -r {DRIVE_DIR} {LOCAL_DIR}
os.chdir(LOCAL_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Isi folder: {os.listdir('.')}")

# Install library
!pip install -q transformers accelerate bitsandbytes Pillow nltk rouge-score tqdm

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PIPELINE LENGKAP: Prepare → Train → Evaluate → Save (satu proses)
# Semua import berada di sel ini agar dapat dijalankan mandiri
# (tahan terhadap reset runtime Colab).
# ═══════════════════════════════════════════════════════════════
import os
import json
import random
import shutil
import warnings
import logging

import torch
import numpy as np
import matplotlib
matplotlib.rcParams["font.size"] = 11
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as calc_meteor
from rouge_score import rouge_scorer
from transformers.utils import logging as hf_logging
import nltk

os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", message=".*MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization.*")
warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
hf_logging.set_verbosity_error()
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

# ─── Konfigurasi ───
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

RANDOM_SEED = 42
IMAGE_SIZE = 384
MAX_LENGTH = 128
NUM_EPOCHS = 20
LR = 2e-5
BATCH_SIZE = 4
MODEL_NAME = "Salesforce/blip2-opt-2.7b"
MODEL_DIR = "model/blip2"
RAW_IMAGE_DIR = "dataset/images"
CAPTION_FILE = "dataset/captions.json"
OUTPUT_DIR = "dataset/processed"
EVAL_DIR = "hasil_evaluasi"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RANDOM_SEED)

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Mapping ltr-XX -> {karakter, vokal}
KARAKTER_DASAR = [
    "a", "ba", "ca", "da", "ga",
    "ha", "ja", "ka", "la", "ma",
    "na", "nga", "nya", "pa", "ra",
    "sa", "ta", "wa", "ya"
]
VOKAL = ["a", "i", "u", "e", "o"]
mapping = {}
_idx = 1
for _dasar in KARAKTER_DASAR:
    for _vokal in VOKAL:
        mapping[f"ltr-{_idx:02d}"] = {"karakter": _dasar, "vokal": _vokal}
        _idx += 1


# ═══════════════════════════════════════════════════════════════
# TAHAP 1: PERSIAPAN DATASET
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 1: PERSIAPAN DATASET")
print("=" * 60)

with open(CAPTION_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Entri caption: {len(raw_data)}")

flat_data = []
for item in raw_data:
    img_id = item["images"]
    img_filename = f"{img_id}.png"
    info = mapping.get(img_id, {})
    for caption in item["captions"]:
        flat_data.append({
            "image": img_filename,
            "caption": caption,
            "karakter": info.get("karakter", ""),
            "vokal": info.get("vokal", ""),
        })
print(f"Total pasang (image, caption): {len(flat_data)}")

# Split per-caption (flat shuffle) — sama dgn training_blip.ipynb
random.shuffle(flat_data)
n = len(flat_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train_data = flat_data[:n_train]
val_data = flat_data[n_train:n_train + n_val]
test_data = flat_data[n_train + n_val:]
print(f"Split: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    os.makedirs(f"{OUTPUT_DIR}/{name}/images", exist_ok=True)
    with open(f"{OUTPUT_DIR}/{name}/captions_{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    processed = set()
    errors = []
    for item in data:
        img = item["image"]
        if img in processed:
            continue
        processed.add(img)
        src = os.path.join(RAW_IMAGE_DIR, img)
        dst = os.path.join(OUTPUT_DIR, name, "images", img)
        if os.path.exists(src):
            try:
                im = Image.open(src).convert("RGB")
                im = im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
                im.save(dst, format="PNG")
                Image.open(dst).verify()
            except Exception as e:
                errors.append(f"{img}: {e}")
        else:
            errors.append(f"{img}: file tidak ditemukan")
    print(f"  {name}: {len(processed)} gambar" + (f" ({len(errors)} error)" if errors else ""))

print("✅ Persiapan dataset selesai!")


# ═══════════════════════════════════════════════════════════════
# TAHAP 2: FINE-TUNING BLIP-2 (Memory-Efficient)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 2: FINE-TUNING BLIP-2")
print("=" * 60)


class Blip2CaptionDataset(Dataset):
    """Dataset khusus untuk BLIP-2."""
    def __init__(self, json_file, image_dir, processor, max_length=128):
        with open(json_file, "r", encoding="utf-8") as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length
        print(f"  Dataset: {len(self.data)} pasang")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.image_dir, os.path.basename(item["image"]))
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (255, 255, 255))
        encoding = self.processor(
            images=image, text=item["caption"],
            padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze()
        pixel_values = encoding["pixel_values"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()
        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }


# Load BLIP-2 dengan 8-bit quantization
print("Memuat BLIP-2 (8-bit)...")
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
print(f"Parameter total: {sum(p.numel() for p in model.parameters()):,}")

# Freeze ViT dan LLM, hanya fine-tune Q-Former dan projection
for param in model.vision_model.parameters():
    param.requires_grad = False
for param in model.language_model.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

# ═══════════════════════════════════════════════════════════════
# PRETRAINED BASELINE EVALUATION (sebelum fine-tuning)
# ═══════════════════════════════════════════════════════════════
print('\n' + '=' * 60)
print('  PRETRAINED BASELINE EVALUATION (sebelum fine-tuning)')
print('=' * 60)

model.eval()

# ─── Peta karakter & regex ekstraksi (dipakai baseline & fine-tuned) ───
import re

CHAR_TO_BASE = {}
for _base in KARAKTER_DASAR:
    if _base == 'a':
        for _v in VOKAL:
            CHAR_TO_BASE[_v] = 'a'
    else:
        _cons = _base[:-1]
        for _v in VOKAL:
            CHAR_TO_BASE[_cons + _v] = _base


def extract_pred_base(caption):
    """Ekstrak karakter dasar dari teks caption via regex."""
    low = caption.lower()
    for pat in [r'yaitu karakter\s+([a-z]+)', r'adalah\s+([a-z]+)\s+dengan',
                r'merupakan karakter\s+([a-z]+)', r'karakter\s+([a-z]+)\s+memiliki',
                r'karakter\s+([a-z]+)\s+dengan']:
        m = re.search(pat, low)
        if m and m.group(1) in CHAR_TO_BASE:
            return CHAR_TO_BASE[m.group(1)]
    return None

with open(f'{OUTPUT_DIR}/test/captions_test.json', 'r', encoding='utf-8') as f:
    bl_test_items = json.load(f)

bl_gambar_dict = {}
for item in bl_test_items:
    key = item['image']
    if key not in bl_gambar_dict:
        bl_gambar_dict[key] = {'captions': [], 'karakter': item.get('karakter', ''), 'vokal': item.get('vokal', '')}
    bl_gambar_dict[key]['captions'].append(item['caption'])

bl_ref = []
bl_hyp = []
bl_detail = []

for image_key, info in tqdm(bl_gambar_dict.items(), desc='Baseline Eval'):
    img_path = os.path.join(f'{OUTPUT_DIR}/test/images', os.path.basename(image_key))
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception:
        continue
    inputs = processor(images=image, return_tensors="pt").to(DEVICE, dtype=torch.float16)
    with torch.no_grad():
        output = eval_model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4,
                                early_stopping=True, no_repeat_ngram_size=2, min_length=10)
    pred = processor.decode(output[0], skip_special_tokens=True).strip()
    refs = [cap.lower().split() for cap in info['captions']]
    hyp = pred.lower().split()
    bl_ref.append(refs)
    bl_hyp.append(hyp)
    bl_detail.append({
        'image': image_key, 'karakter': info['karakter'], 'vokal': info['vokal'],
        'caption_prediksi': pred, 'captions_referensi': info['captions']
    })

bl_smoother = SmoothingFunction().method1
baseline_metrik = {
    'BLEU-1': corpus_bleu(bl_ref, bl_hyp, weights=(1,0,0,0), smoothing_function=bl_smoother) * 100,
    'BLEU-2': corpus_bleu(bl_ref, bl_hyp, weights=(0.5,0.5,0,0), smoothing_function=bl_smoother) * 100,
    'BLEU-3': corpus_bleu(bl_ref, bl_hyp, weights=(0.33,0.33,0.33,0), smoothing_function=bl_smoother) * 100,
    'BLEU-4': corpus_bleu(bl_ref, bl_hyp, weights=(0.25,0.25,0.25,0.25), smoothing_function=bl_smoother) * 100,
    'METEOR': np.mean([calc_meteor(refs, hyp) for refs, hyp in zip(bl_ref, bl_hyp)]) * 100,
    'ROUGE-L': np.mean([max(rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False).score(' '.join(r), ' '.join(hyp))['rougeL'].fmeasure for r in refs) for refs, hyp in zip(bl_ref, bl_hyp)]) * 100,
}

bl_karakter_stats = {}
for item in bl_detail:
    kar = item['karakter']
    if not kar:
        continue
    bl_karakter_stats.setdefault(kar, {'total': 0, 'benar': 0})
    bl_karakter_stats[kar]['total'] += 1
    if extract_pred_base(item['caption_prediksi']) == kar:
        bl_karakter_stats[kar]['benar'] += 1

bl_total_benar = sum(s['benar'] for s in bl_karakter_stats.values())
bl_total_semua = sum(s['total'] for s in bl_karakter_stats.values())
bl_akurasi = (bl_total_benar / bl_total_semua * 100) if bl_total_semua > 0 else 0

print('\n' + '=' * 60)
print('  HASIL PRETRAINED BASELINE — TEST SET')
print('=' * 60)
print('┌────────────┬──────────┐')
print('│   Metrik   │   Skor   │')
print('├────────────┼──────────┤')
for k, v in baseline_metrik.items():
    print(f'│ {k:<10} │  {v:>5.2f}%  │')
print('└────────────┴──────────┘')
print(f'\n  Akurasi Identifikasi Karakter (Baseline): {bl_akurasi:.1f}% ({bl_total_benar}/{bl_total_semua})')

# ─── Baseline Confusion Matrix ───
bl_cm_labels = sorted({d['karakter'] for d in bl_detail if d['karakter']})
bl_cm_lidx = {l: i for i, l in enumerate(bl_cm_labels)}
bl_cm = np.zeros((len(bl_cm_labels), len(bl_cm_labels) + 1), dtype=int)
for item in bl_detail:
    kar = item['karakter']
    if not kar:
        continue
    pb = extract_pred_base(item['caption_prediksi'])
    j = bl_cm_lidx.get(pb, len(bl_cm_labels))
    bl_cm[bl_cm_lidx[kar], j] += 1

bl_cm_xlabels = bl_cm_labels + ['?']
fig, ax = plt.subplots(figsize=(max(8, len(bl_cm_labels) * 0.6), max(7, len(bl_cm_labels) * 0.55)))
im = ax.imshow(bl_cm, cmap='Reds')
ax.set_xticks(range(len(bl_cm_xlabels)))
ax.set_xticklabels(bl_cm_xlabels, rotation=45, ha='right')
ax.set_yticks(range(len(bl_cm_labels)))
ax.set_yticklabels(bl_cm_labels)
ax.set_xlabel('Prediksi')
ax.set_ylabel('Aktual')
ax.set_title('Confusion Matrix — Pretrained Baseline (BLIP-2)', fontweight='bold')
_bl_thr = bl_cm.max() / 2 if bl_cm.max() > 0 else 0
for _i in range(bl_cm.shape[0]):
    for _j in range(bl_cm.shape[1]):
        if bl_cm[_i, _j] > 0:
            ax.text(_j, _i, bl_cm[_i, _j], ha='center', va='center',
                    color='white' if bl_cm[_i, _j] > _bl_thr else 'black', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
os.makedirs(EVAL_DIR, exist_ok=True)
plt.savefig(f'{EVAL_DIR}/confusion_matrix_blip2_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Baseline confusion matrix disimpan: {EVAL_DIR}/confusion_matrix_blip2_baseline.png')

# Simpan baseline results
baseline_hasil = {
    'metrik': {k: round(v, 2) for k, v in baseline_metrik.items()},
    'akurasi_per_karakter': {k: round((v['benar']/v['total']*100) if v['total']>0 else 0, 1)
                             for k, v in bl_karakter_stats.items()},
    'akurasi_total': round(bl_akurasi, 1),
    'detail': bl_detail
}
with open(f'{EVAL_DIR}/baseline_blip2.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_hasil, f, ensure_ascii=False, indent=2)
print(f'✅ Baseline disimpan: {EVAL_DIR}/baseline_blip2.json')


# Gradient checkpointing
model.gradient_checkpointing_enable()

# Simpan processor
processor.save_pretrained(f"{MODEL_DIR}/best_model")

# DataLoader
train_ds = Blip2CaptionDataset(f"{OUTPUT_DIR}/train/captions_train.json", f"{OUTPUT_DIR}/train/images", processor, MAX_LENGTH)
val_ds = Blip2CaptionDataset(f"{OUTPUT_DIR}/val/captions_val.json", f"{OUTPUT_DIR}/val/images", processor, MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(DEVICE == "cuda"))

# Training (mixed precision manual)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler()

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Train", leave=False, mininterval=2.0):
        pixel_values = batch["pixel_values"].to(DEVICE, dtype=torch.float16)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} Val", leave=False, mininterval=2.0):
            pixel_values = batch["pixel_values"].to(DEVICE, dtype=torch.float16)
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast():
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
            total_loss += outputs.loss.item()
    val_loss = total_loss / len(val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(f"{MODEL_DIR}/best_model", exist_ok=True)
        model.save_pretrained(f"{MODEL_DIR}/best_model")
        print(f"  -> Best model! Val Loss: {best_val_loss:.4f}")

    if epoch % 5 == 0:
        ckpt = f"{MODEL_DIR}/checkpoint_epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)

with open(f"{MODEL_DIR}/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"\nTraining selesai! Best Val Loss: {best_val_loss:.4f}")


# ═══════════════════════════════════════════════════════════════
# TAHAP 3: EVALUASI PADA TEST SET
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 3: EVALUASI BLIP-2")
print("=" * 60)

# Plot kurva training (Train vs Val Loss)
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
ax.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#FF9800")
ax.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#FF5722")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("BLIP-2 - Training & Validation Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/training_curve_blip2.png", dpi=150, bbox_inches="tight")
plt.show()

# Evaluasi menggunakan best_model dari disk (best practice)
print(f"Memuat best model dari: {MODEL_DIR}/best_model")
try:
    eval_model = Blip2ForConditionalGeneration.from_pretrained(
        f"{MODEL_DIR}/best_model",
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    eval_model.eval()
except Exception as e:
    print(f"Gagal memuat best_model dari disk ({e}), fallback ke model in-memory.")
    eval_model = model
    eval_model.eval()

with open(f"{OUTPUT_DIR}/test/captions_test.json", "r", encoding="utf-8") as f:
    test_items = json.load(f)
print(f"Data test: {len(test_items)} entri")

# Kelompokkan per gambar + ambil karakter/vokal dari mapping
gambar_dict = {}
for item in test_items:
    key = item["image"]
    if key not in gambar_dict:
        ltr_id = os.path.splitext(os.path.basename(key))[0]
        info = mapping.get(ltr_id, {})
        gambar_dict[key] = {"captions": [], "karakter": info.get("karakter", ""), "vokal": info.get("vokal", "")}
    gambar_dict[key]["captions"].append(item["caption"])

# Generate & kumpulkan referensi/hipotesis
referensi_list = []
hipotesis_list = []
hasil_detail = []
for image_key, info in tqdm(gambar_dict.items(), desc="Evaluasi Test Set"):
    img_path = os.path.join(f"{OUTPUT_DIR}/test/images", os.path.basename(image_key))
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception:
        continue
    inputs = processor(images=image, return_tensors="pt").to(DEVICE, dtype=torch.float16)
    with torch.no_grad():
        output = eval_model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4,
                                early_stopping=True, no_repeat_ngram_size=2, min_length=10)
    pred = processor.decode(output[0], skip_special_tokens=True).strip()
    refs = [cap.lower().split() for cap in info["captions"]]
    hyp = pred.lower().split()
    referensi_list.append(refs)
    hipotesis_list.append(hyp)
    hasil_detail.append({
        "image": image_key, "karakter": info["karakter"], "vokal": info["vokal"],
        "caption_prediksi": pred, "captions_referensi": info["captions"]
    })

# Hitung metrik (BLEU-1..4, METEOR, ROUGE-L)
smoother = SmoothingFunction().method1
bleu1 = corpus_bleu(referensi_list, hipotesis_list, weights=(1, 0, 0, 0), smoothing_function=smoother)
bleu2 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.5, 0.5, 0, 0), smoothing_function=smoother)
bleu3 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoother)
bleu4 = corpus_bleu(referensi_list, hipotesis_list, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)
meteor_avg = np.mean([calc_meteor(refs, hyp) for refs, hyp in zip(referensi_list, hipotesis_list)])
rouge_sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
rouge_scores_list = []
for refs, hyp in zip(referensi_list, hipotesis_list):
    hyp_str = " ".join(hyp)
    rouge_scores_list.append(max(rouge_sc.score(" ".join(r), hyp_str)["rougeL"].fmeasure for r in refs))
rouge_avg = np.mean(rouge_scores_list)

metrik = {
    "BLEU-1": bleu1 * 100, "BLEU-2": bleu2 * 100,
    "BLEU-3": bleu3 * 100, "BLEU-4": bleu4 * 100,
    "METEOR": meteor_avg * 100, "ROUGE-L": rouge_avg * 100
}

print("\n" + "=" * 60)
print("  HASIL EVALUASI — TEST SET")
print("=" * 60)
print("┌────────────┬──────────┐")
print("│   Metrik   │   Skor   │")
print("├────────────┼──────────┤")
for k, v in metrik.items():
    print(f"│ {k:<10} │  {v:>5.2f}%  │")
print("└────────────┴──────────┘")

# Akurasi identifikasi per karakter dasar (regex-based)
print("\n  Akurasi Identifikasi per Karakter Dasar:")
karakter_stats = {}
for item in hasil_detail:
    kar = item["karakter"]
    if not kar:
        continue
    karakter_stats.setdefault(kar, {"total": 0, "benar": 0})
    karakter_stats[kar]["total"] += 1
    if extract_pred_base(item["caption_prediksi"]) == kar:
        karakter_stats[kar]["benar"] += 1

print("┌──────────────┬───────┬────────┬──────────┐")
print("│   Karakter   │ Total │ Benar  │ Akurasi  │")
print("├──────────────┼───────┼────────┼──────────┤")
total_benar = 0
total_semua = 0
for kar in sorted(karakter_stats.keys()):
    s = karakter_stats[kar]
    akurasi = (s["benar"] / s["total"] * 100) if s["total"] > 0 else 0
    total_benar += s["benar"]
    total_semua += s["total"]
    print(f"│ {kar:<12} │  {s['total']:>3}  │   {s['benar']:>3}  │  {akurasi:>5.1f}%  │")
print("├──────────────┼───────┼────────┼──────────┤")
akurasi_total = (total_benar / total_semua * 100) if total_semua > 0 else 0
print(f"│ {'TOTAL':<12} │  {total_semua:>3}  │   {total_benar:>3}  │  {akurasi_total:>5.1f}%  │")
print("└──────────────┴───────┴────────┴──────────┘")

# Visualisasi metrik + akurasi per karakter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
metrik_names = list(metrik.keys())
metrik_vals = list(metrik.values())
bars = ax1.bar(metrik_names, metrik_vals,
               color=["#2196F3", "#2196F3", "#2196F3", "#2196F3", "#4CAF50", "#FF9800"])
ax1.set_ylabel("Score (%)")
ax1.set_title("Evaluation Metrics — Test Set (BLIP-2)")
ax1.set_ylim(0, 100)
ax1.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, metrik_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
if karakter_stats:
    kar_names = sorted(karakter_stats.keys())
    kar_akurasi = [(karakter_stats[k]["benar"] / karakter_stats[k]["total"] * 100)
                   if karakter_stats[k]["total"] > 0 else 0 for k in kar_names]
    bar_colors = ["#4CAF50" if a >= 80 else "#FF9800" if a >= 50 else "#F44336" for a in kar_akurasi]
    ax2.bar(kar_names, kar_akurasi, color=bar_colors)
    ax2.set_ylabel("Akurasi (%)")
    ax2.set_title("Akurasi Identifikasi per Karakter Dasar")
    ax2.set_ylim(0, 105)
    ax2.axhline(y=80, color="green", linestyle="--", alpha=0.5, label="Target 80%")
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/evaluasi_metrik_blip2.png", dpi=150, bbox_inches="tight")
plt.show()

# ─── Confusion Matrix (identifikasi karakter dasar) ───
cm_labels = sorted({d["karakter"] for d in hasil_detail if d["karakter"]})
cm_lidx = {l: i for i, l in enumerate(cm_labels)}
cm = np.zeros((len(cm_labels), len(cm_labels) + 1), dtype=int)
for item in hasil_detail:
    kar = item["karakter"]
    if not kar:
        continue
    pb = extract_pred_base(item["caption_prediksi"])
    j = cm_lidx.get(pb, len(cm_labels))
    cm[cm_lidx[kar], j] += 1
cm_xlabels = cm_labels + ["?"]  # "?" = prediksi tak terbaca (regex gagal)
fig, ax = plt.subplots(figsize=(max(8, len(cm_labels) * 0.6), max(7, len(cm_labels) * 0.55)))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(cm_xlabels)))
ax.set_xticklabels(cm_xlabels, rotation=45, ha="right")
ax.set_yticks(range(len(cm_labels)))
ax.set_yticklabels(cm_labels)
ax.set_xlabel("Prediksi")
ax.set_ylabel("Aktual")
ax.set_title("Confusion Matrix — Karakter Dasar (BLIP-2)", fontweight="bold")
_thr = cm.max() / 2 if cm.max() > 0 else 0
for _i in range(cm.shape[0]):
    for _j in range(cm.shape[1]):
        if cm[_i, _j] > 0:
            ax.text(_j, _i, cm[_i, _j], ha="center", va="center",
                    color="white" if cm[_i, _j] > _thr else "black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(f"{EVAL_DIR}/confusion_matrix_blip2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix disimpan:", f"{EVAL_DIR}/confusion_matrix_blip2.png")

# ─── Perbandingan: Pretrained Baseline vs Fine-tuned ───
print('\n' + '=' * 60)
print('  PERBANDINGAN: PRETRAINED BASELINE vs FINE-TUNED')
print('=' * 60)
print('┌────────────┬──────────────┬──────────────┬───────────┐')
print('│   Metrik   │  Pretrained  │  Fine-tuned  │ Delta (%) │')
print('├────────────┼──────────────┼──────────────┼───────────┤')
for k in metrik.keys():
    bl_val = baseline_metrik.get(k, 0)
    ft_val = metrik[k]
    delta = ft_val - bl_val
    print(f'│ {k:<10} │  {bl_val:>7.2f}%   │  {ft_val:>7.2f}%   │  {delta:>+6.2f}%  │')
print('└────────────┴──────────────┴──────────────┴───────────┘')

ft_total_benar = sum(s['benar'] for s in karakter_stats.values())
ft_total_semua = sum(s['total'] for s in karakter_stats.values())
ft_akurasi = (ft_total_benar / ft_total_semua * 100) if ft_total_semua > 0 else 0
print(f'\n  Akurasi Identifikasi Karakter:')
print(f'    Pretrained : {bl_akurasi:.1f}% ({bl_total_benar}/{bl_total_semua})')
print(f'    Fine-tuned : {ft_akurasi:.1f}% ({ft_total_benar}/{ft_total_semua})')
print(f'    Delta      : {ft_akurasi - bl_akurasi:+.1f}%')

fig, ax = plt.subplots(figsize=(12, 6))
metric_names = list(metrik.keys())
x = np.arange(len(metric_names))
width = 0.35
bl_vals = [baseline_metrik.get(k, 0) for k in metric_names]
ft_vals = [metrik[k] for k in metric_names]
ax.bar(x - width/2, bl_vals, width, label='Pretrained Baseline', color='#FF7043', alpha=0.8)
ax.bar(x + width/2, ft_vals, width, label='Fine-tuned', color='#42A5F5', alpha=0.8)
ax.set_xlabel('Metrik')
ax.set_ylabel('Score (%)')
ax.set_title('Perbandingan: Pretrained Baseline vs Fine-tuned (BLIP-2)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 100)
for i, (bl_v, ft_v) in enumerate(zip(bl_vals, ft_vals)):
    ax.text(i - width/2, bl_v + 1, f'{bl_v:.1f}%', ha='center', va='bottom', fontsize=8)
    ax.text(i + width/2, ft_v + 1, f'{ft_v:.1f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/perbandingan_baseline_vs_finetuned_blip2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Perbandingan disimpan: {EVAL_DIR}/perbandingan_baseline_vs_finetuned_blip2.png')


# Simpan hasil evaluasi lengkap
hasil_final = {
    "metrik": {k: round(v, 2) for k, v in metrik.items()},
    "akurasi_per_karakter": {k: round((v["benar"] / v["total"] * 100) if v["total"] > 0 else 0, 1)
                             for k, v in karakter_stats.items()},
    "training_history": {
        "best_val_loss": round(best_val_loss, 4),
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"]
    },
    "detail": hasil_detail
}
with open(f"{EVAL_DIR}/evaluasi_blip2.json", "w", encoding="utf-8") as f:
    json.dump(hasil_final, f, ensure_ascii=False, indent=2)
print(f"\nHasil evaluasi disimpan: {EVAL_DIR}/evaluasi_blip2.json")


# ═══════════════════════════════════════════════════════════════
# TAHAP 4: INFERENCE & VISUALISASI (contoh gambar test)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 4: INFERENCE & VISUALISASI")
print("=" * 60)
test_folder = f"{OUTPUT_DIR}/test/images"
test_images = sorted([f for f in os.listdir(test_folder) if f.endswith(".png")])[:6]
inference_results = []
n_show = len(test_images)
if n_show > 0:
    rows = 2 if n_show > 3 else 1
    cols = int(np.ceil(n_show / rows))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).flatten()
    for idx, img_name in enumerate(test_images):
        img_path = os.path.join(test_folder, img_name)
        image = Image.open(img_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(DEVICE, dtype=torch.float16)
        with torch.no_grad():
            output = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=4,
                                    early_stopping=True, no_repeat_ngram_size=2, min_length=10)
        caption = processor.decode(output[0], skip_special_tokens=True).strip()
        inference_results.append({"gambar": img_name, "caption": caption})
        print(f"  {img_name} → {caption}")
        axes[idx].imshow(image)
        axes[idx].set_title(caption, fontsize=9, wrap=True)
        axes[idx].axis("off")
    for idx in range(n_show, len(axes)):
        axes[idx].axis("off")
    plt.tight_layout()
    plt.savefig(f"{EVAL_DIR}/visualisasi_prediksi_blip2.png", dpi=150, bbox_inches="tight")
    plt.show()
with open(f"{EVAL_DIR}/hasil_inference_blip2.json", "w", encoding="utf-8") as f:
    json.dump(inference_results, f, ensure_ascii=False, indent=2)


# ═══════════════════════════════════════════════════════════════
# TAHAP 5: SIMPAN KE GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  TAHAP 5: SIMPAN KE GOOGLE DRIVE")
print("=" * 60)

DRIVE_DIR = "/content/drive/MyDrive/ImageCaptioning"
LOCAL_DIR = "/content/ImageCaptioning"

for folder in [MODEL_DIR, EVAL_DIR]:
    src = os.path.join(LOCAL_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {folder}/ -> Drive")

print("\n✅ SELURUH PIPELINE SELESAI! Semua hasil tersimpan di Google Drive.")